# Honey, I Shrunk the Diffusion: sample inference

Runs the four models from the report on one SID test image and shows them side by side:

| Key | Model |
|---|---|
| `sd21` | SD 2.1, DarkDiff baseline (no illumination conditioning) |
| `bk-sdm-v2-tiny` | BK-SDM-v2-tiny + illumination conditioning |
| `tiny-sd` | Tiny-SD + illumination conditioning |
| `sdxs-512` | SDXS-512 + illumination conditioning |

**Before running**
1. Settings → Accelerator: a GPU. Settings → Internet: **on** (the code is cloned from GitHub and the
   pretrained U-Nets, CLIP text encoders and SD VAE come from the Hugging Face Hub).
2. Add the dataset with the checkpoints and the sample folder (made on the server with `export_sample.py`).
3. Add the HISTD dataset as an input: https://www.kaggle.com/datasets/shambac/histd-ckpts
4. Fill in the paths in the **Settings** cell.

## Install dependencies

In [ ]:
%pip install -q "diffusers>=0.42" "transformers>=5.20" rawpy lpips torchmetrics

## Get the code

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/ShambaC/Honey-I-Shrunk-The-DIffusion.git"
REPO_DIR = "/kaggle/working/DarkDiff"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)  # pick up pushed fixes
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
# first on the path, so the repo's evaluate.py wins over Hugging Face's `evaluate` package
sys.path.insert(0, REPO_DIR)

## Settings

In [ ]:
from pathlib import Path

# Stage-2 checkpoints (ldm_last.pt from each run), e.g. "/kaggle/input/<dataset>/sd21/ldm_last.pt".
# Illumination conditioning is detected from the checkpoint keys. Models left blank are skipped.
CKPTS = {
    "sd21": "/kaggle/input/datasets/shambac/histd-ckpts/sd21/ldm/ldm_last.pt",
    "bk-sdm-v2-tiny": "/kaggle/input/datasets/shambac/histd-ckpts/bk-sdm-v2-tiny-illum/ldm/ldm_last.pt",
    "tiny-sd": "/kaggle/input/datasets/shambac/histd-ckpts/tiny-sd-illum/ldm/ldm_last.pt",
    "sdxs-512": "/kaggle/input/datasets/shambac/histd-ckpts/sdxs-512-illum/ldm/ldm_last.pt",
}
LABELS = {
    "sd21": "SD 2.1 (baseline)",
    "bk-sdm-v2-tiny": "BK-SDM-v2-tiny + illum.",
    "tiny-sd": "Tiny-SD + illum.",
    "sdxs-512": "SDXS-512 + illum.",
}

# Folder written by export_sample.py: pair.json plus the noisy and ground-truth .ARW files
SAMPLE_DIR = "/kaggle/input/datasets/shambac/histd-ckpts/sample"

CROP = 1024    # centre crop (px, half-resolution image) to fit a 16 GB GPU; None = full 2128x1424 frame
STEPS = 50     # DDIM steps, as in the report
GUIDANCE = 2.0 # CFG weight for SID, as in the report
SEED = 0       # same starting noise for every model
OUT_DIR = Path("/kaggle/working/samples")

## Imports

In [ ]:
import gc
import json

import lpips
import matplotlib.pyplot as plt
import torch
from torchmetrics.functional.image import peak_signal_noise_ratio, structural_similarity_index_measure
from torchvision.utils import save_image

from darkdiff.data import PairDataset
from evaluate import load_model

dev = "cuda" if torch.cuda.is_available() else "cpu"
# bf16 needs Ampere or newer (as on the A40 used for training); Kaggle's T4 / P100 fall back to fp32
use_bf16 = dev == "cuda" and torch.cuda.get_device_capability()[0] >= 8
print(dev, torch.cuda.get_device_name() if dev == "cuda" else "", "bf16" if use_bf16 else "fp32")

## Load the sample

In [ ]:
assert SAMPLE_DIR, "set SAMPLE_DIR in the Settings cell"
sample = Path(SAMPLE_DIR)
meta = json.loads((sample / "pair.json").read_text())
item = PairDataset([(str(sample / meta["noisy"]), str(sample / meta["gt"]), meta["ratio"])], None, train=False)[0]
noisy, srgb = item["noisy"][None], item["srgb"][None]  # [-1, 1], shape 1x3xHxW

if CROP:
    H, W = noisy.shape[-2:]
    c = min(CROP, H, W) // 8 * 8
    y, x = (H - c) // 2, (W - c) // 2
    noisy, srgb = noisy[..., y:y + c, x:x + c], srgb[..., y:y + c, x:x + c]

gt = (srgb + 1) / 2
# the input is linear RGB after the x-ratio gain: gamma 2.2 for display only
noisy_show = ((noisy + 1) / 2).clamp(0, 1) ** (1 / 2.2)
OUT_DIR.mkdir(parents=True, exist_ok=True)
save_image(noisy_show, OUT_DIR / "input.png")
save_image(gt, OUT_DIR / "reference.png")
print(f"{meta['noisy']} (x{meta['ratio']:g}), image {tuple(noisy.shape[-2:])}")

## Run the four models

In [ ]:
lpips_fn = lpips.LPIPS(net="alex").to(dev).eval()
results = {}

for name, ckpt in CKPTS.items():
    if not ckpt:
        print(f"{name}: no checkpoint path, skipped")
        continue
    state = torch.load(ckpt, map_location="cpu", weights_only=True)
    illum = any(".film." in k for k in state)  # same check as sample_images.py
    model = load_model(name, state, illum=illum).to(dev).eval()
    del state

    torch.manual_seed(SEED)
    with torch.no_grad(), torch.autocast(dev, dtype=torch.bfloat16, enabled=use_bf16):
        pred = model.sample(noisy.to(dev), steps=STEPS, guidance=GUIDANCE)
    pred = ((pred.float() + 1) / 2).cpu()

    with torch.no_grad():
        m = {
            "psnr": peak_signal_noise_ratio(pred, gt, data_range=1.0).item(),
            "ssim": structural_similarity_index_measure(pred, gt, data_range=1.0).item(),
            "lpips": lpips_fn(pred.to(dev) * 2 - 1, gt.to(dev) * 2 - 1).mean().item(),
        }
    save_image(pred, OUT_DIR / f"{name}.png")
    results[name] = (pred, m)
    print(f"{name} ({'illum' if illum else 'vanilla'}): "
          f"PSNR {m['psnr']:.2f} dB, SSIM {m['ssim']:.4f}, LPIPS {m['lpips']:.4f}")

    del model, pred
    gc.collect()
    torch.cuda.empty_cache()

## Compare

In [ ]:
def show(ax, img, title):
    ax.imshow(img[0].permute(1, 2, 0).clamp(0, 1).numpy())
    ax.set_title(title, fontsize=10)
    ax.axis("off")


panels = [(noisy_show, "Input, short exposure"), (gt, "Reference, long exposure")]
panels += [(p, f"{LABELS[n]}\nPSNR {m['psnr']:.2f} | SSIM {m['ssim']:.3f} | LPIPS {m['lpips']:.3f}")
           for n, (p, m) in results.items()]

cols = 3
rows = -(-len(panels) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, (img, title) in zip(axes.flat, panels):
    show(ax, img, title)
fig.tight_layout()
fig.savefig(OUT_DIR / "comparison.png", dpi=150)
plt.show()
print(f"images saved to {OUT_DIR}")